# **Project Name**    - Unsupervised ML - Netflix Movies & TV Shows Clustering

##### **Project Type**    - Unsupervised Learning (Clustering) + Exploratory Data Analysis
##### **Contribution**    - Individual
##### **Team Member 1 -** [Your Name Here]


# **Project Summary -**

This project analyzes a catalog of 7,787 Movies and TV Shows available on Netflix as of 2019, originally
collected from Flixable (a third-party Netflix search engine). Flixable's 2018 report found that Netflix's TV show
catalog had nearly tripled since 2010, while its movie catalog had shrunk by more than 2,000 titles over the same
period — one motivation for this analysis.

The notebook begins with an exploratory data analysis (EDA) covering the mix of Movies vs. TV Shows over time,
the countries producing the most content, popular genres and ratings, typical movie/TV-show durations, and the
most frequently credited directors and cast members. Three formal hypotheses are then tested statistically:
(1) whether Netflix's addition of TV shows relative to movies has increased in recent years, (2) whether there is a
statistically significant association between a title's country of origin and whether its content rating is "mature",
and (3) whether average movie duration differs significantly across rating categories. All three tests returned
statistically significant results (p < 0.01), confirming Netflix's growing TV focus, meaningful country-level rating
differences, and duration differences by rating.

After cleaning missing values (`director`, `cast`, `country` were imputed as `'Unknown'` rather than dropped, since
even titles with missing credits still carry a usable genre/description for clustering) and engineering features
(`year_added`, `duration_int`, `primary_country`, `content_age_at_add`, and a combined text "soup" of genre +
description + director + cast + type), the notebook builds a full **text-clustering pipeline**: TF-IDF vectorization
(with a lightweight, dependency-free contraction-expansion, punctuation/URL removal, stop-word removal, and
rule-based stemming step, since NLTK/contraction packages are not available in this offline environment),
TruncatedSVD dimensionality reduction, and L2-row normalization (a standard "spherical k-means" trick that makes
Euclidean distance behave like cosine similarity for text).

Three clustering algorithms were then trained and compared: **K-Means** (k chosen via the Elbow method and
Silhouette score), **Agglomerative Clustering** (validated with a dendrogram and compared across linkage methods),
and **DBSCAN** (tuned via a grid search over `eps`/`min_samples`, useful for flagging outlier/noise titles).
K-Means produced the best balance of silhouette score (~0.18) and cluster interpretability, yielding 10 clusters that
map cleanly onto recognizable content themes — stand-up comedy specials, horror/thriller movies, children & family
films, kids'/reality TV, action & adventure, independent dramas, documentaries, romance, international TV
dramas/crime shows, and India-heavy international dramas/comedies. Each cluster is profiled by its most distinctive
TF-IDF terms and its dominant content type, country, and rating, and the notebook closes with concrete, cluster-level
recommendations for content acquisition, regional licensing, and personalized recommendation strategy — the kind of
segmentation a streaming platform's content and marketing teams could act on directly.


# **GitHub Link -**

Add your GitHub repository link here once you push this notebook and any supporting files.

# **Problem Statement**


Netflix's content catalog has grown rapidly and diversely since 2010, with a well-documented shift toward TV
shows and away from movies. Beyond that high-level trend, Netflix's catalog contains thousands of titles spanning
many countries, genres, and ratings with no pre-existing labels grouping similar content together.

**The problem:** given only descriptive metadata (genre tags, plot description, cast, director, content type,
country, rating, duration) and no target/label column, can we automatically discover meaningful, natural groupings
("clusters") of similar Netflix titles? Such clusters could power content recommendations ("more like this"),
reveal gaps in the catalog (genres/countries that are under-represented), and give content and marketing teams a
data-driven way to segment the catalog for regional or personalization strategy — without manually tagging
thousands of titles.


#### **Define Your Business Objective?**

The business objective is to **segment the Netflix catalog into content clusters using unsupervised learning**,
so that:
1. **Content/recommendation teams** can improve "Because you watched…" style recommendations by surfacing titles
   from the same content cluster.
2. **Content acquisition & production teams** can identify catalog gaps — clusters that are small, or under-represented
   in certain countries — and prioritize licensing/production accordingly.
3. **Regional/marketing teams** can understand which content themes dominate in which countries, to guide localization
   and regional marketing campaigns.

This is a pure unsupervised learning problem: there is no ground-truth label to predict, so success is measured via
clustering-quality metrics (Silhouette Score, Davies–Bouldin Index) and, more importantly, via whether the resulting
clusters are **interpretable and actionable** for the stakeholders above.


# **General Guidelines** : -  

1.   Well-structured, formatted, and commented code is required.
2.   Exception Handling, Production Grade Code & Deployment Ready Code will be a plus. Those students will be awarded some additional credits.
     
     The additional credits will have advantages over other students during Star Student selection.
       
             [ Note: - Deployment Ready Code is defined as, the whole .ipynb notebook should be executable in one go
                       without a single error logged. ]

3.   Each and every logic should have proper comments.
4. You may add as many number of charts you want. Make Sure for each and every chart the following format should be answered.
        

```
# Chart visualization code
```
            

*   Why did you pick the specific chart?
*   What is/are the insight(s) found from the chart?
* Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

5. You have to create at least 20 logical & meaningful charts having important insights.


[ Hints : - Do the Vizualization in  a structured way while following "UBM" Rule.

U - Univariate Analysis,

B - Bivariate Analysis (Numerical - Categorical, Numerical - Numerical, Categorical - Categorical)

M - Multivariate Analysis
 ]





# ***Let's Begin !***

## ***1. Know Your Data***

### Import Libraries

In [ ]:
# Import Libraries
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from scipy import stats

import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 5)
NETFLIX_RED = '#E50914'
NETFLIX_BLACK = '#221f1f'


### Dataset Loading

In [ ]:
# Load Dataset
df = pd.read_csv('netflix_titles.csv')


### Dataset First View

In [ ]:
# Dataset First Look
df.head()


### Dataset Rows & Columns count

In [ ]:
# Dataset Rows & Columns count
print("Number of rows:", df.shape[0])
print("Number of columns:", df.shape[1])


### Dataset Information

In [ ]:
# Dataset Info
df.info()


#### Duplicate Values

In [ ]:
# Dataset Duplicate Value Count
print("Fully duplicate rows:", df.duplicated().sum())
print("Duplicate titles:", df['title'].duplicated().sum())
print("Duplicate show_id:", df['show_id'].duplicated().sum())


#### Missing Values/Null Values

In [ ]:
# Missing Values/Null Values Count
missing = df.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(df) * 100).round(2)
missing_df = pd.DataFrame({'missing_count': missing, 'missing_pct (%)': missing_pct})
missing_df[missing_df.missing_count > 0]


In [ ]:
# Visualizing the missing values
plt.figure(figsize=(10, 5))
sns.heatmap(df.isnull(), cbar=False, cmap='rocket')
plt.title('Missing Value Map')
plt.show()


### What did you know about your dataset?

The dataset has **7,787 rows and 12 columns**, describing Netflix Movies and TV Shows available as of 2019
(sourced from Flixable). There are **no fully duplicate rows or duplicate titles**. Four columns have missing
values: `director` (~30.7% missing), `cast` (~9.2%), `country` (~6.5%), and small amounts of missing `date_added`
(10 rows) and `rating` (7 rows). `director`/`cast`/`country` are missing because Netflix's own metadata does not
always list them (e.g., for stand-up specials or documentaries with no formal "cast"), not because of a data
collection error — so these will be imputed as `'Unknown'` rather than dropped, to avoid losing thousands of usable
rows.


## ***2. Understanding Your Variables***

In [ ]:
# Dataset Columns
df.columns.tolist()


In [ ]:
# Dataset Describe
df.describe(include='all').T


### Variables Description

- **show_id**: unique identifier for each title.
- **type**: `Movie` or `TV Show`.
- **title**: name of the title.
- **director**: director(s) of the title (missing for ~31% of rows, mostly TV shows/specials with no single director).
- **cast**: comma-separated list of cast members (missing for ~9%).
- **country**: comma-separated list of countries of production (missing for ~6.5%).
- **date_added**: date the title was added to Netflix (as a string, e.g. "September 9, 2019").
- **release_year**: the year the title was originally released.
- **rating**: content/age rating (e.g. `TV-MA`, `PG-13`).
- **duration**: either `"X min"` (Movies) or `"X Season(s)"` (TV Shows) — a mixed-unit text field.
- **listed_in**: comma-separated genre tags (e.g. `"Dramas, International Movies"`).
- **description**: a short plot synopsis of the title.


### Check Unique Values for each variable.

In [ ]:
# Check Unique Values for each variable.
for col in df.columns:
    print(f"{col}: {df[col].nunique()} unique values")


## 3. ***Data Wrangling***

### Data Wrangling Code

In [ ]:
# Write your code to make your dataset analysis ready.

def clean_data(data):
    data = data.copy()
    # Categorical/text columns: missing usually means "not credited", not an error -> fill 'Unknown'
    for col in ['director', 'cast', 'country']:
        data[col] = data[col].fillna('Unknown')
        data[col] = data[col].replace('', 'Unknown')
    # Rating: only 7 missing rows -> impute with the mode
    data['rating'] = data['rating'].fillna(data['rating'].mode()[0])
    # date_added: only 10 missing rows -> safe to drop
    data = data.dropna(subset=['date_added'])
    # Remove any exact duplicate rows
    data = data.drop_duplicates()
    return data.reset_index(drop=True)


def engineer_features(data):
    data = data.copy()
    # Parse the date Netflix added the title
    data['date_added_parsed'] = pd.to_datetime(data['date_added'].str.strip(), errors='coerce')
    data['year_added'] = data['date_added_parsed'].dt.year
    data['month_added'] = data['date_added_parsed'].dt.month

    # Split the mixed-unit `duration` column into a type flag + numeric value
    data['duration_type'] = np.where(data['duration'].str.contains('Season', case=False, na=False),
                                      'Season(s)', 'Minutes')
    data['duration_int'] = data['duration'].str.extract(r'(\d+)').astype(float)

    # First-listed country / genre, useful for grouping and plotting
    data['primary_country'] = data['country'].apply(lambda x: str(x).split(',')[0].strip())
    data['primary_genre'] = data['listed_in'].apply(lambda x: str(x).split(',')[0].strip())

    # How many years after release Netflix added the title (content "freshness"); clip negative
    # values (a handful of data-entry quirks where release_year > year_added) at 0
    data['content_age_at_add'] = (data['year_added'] - data['release_year']).clip(lower=0)

    return data

df = clean_data(df)
df = engineer_features(df)
print("Shape after cleaning + feature engineering:", df.shape)
df[['title', 'duration_type', 'duration_int', 'primary_country', 'primary_genre', 'content_age_at_add']].head()


### What all manipulations have you done and insights you found?

- **Missing-value handling:** `director`, `cast`, and `country` were filled with `'Unknown'` instead of being
  dropped — dropping them would have discarded up to ~31% of rows, and these titles still carry usable genre and
  description text for later clustering. `rating` (7 missing) was imputed with its mode, and the 10 rows missing
  `date_added` were dropped since that's a negligible fraction of the data and there's no sensible way to impute a
  specific date.
- **Feature engineering insights:** after parsing `date_added`, we can see Netflix's per-year addition volume
  peaked in 2019-2020, and splitting `duration` into a numeric value revealed movies average about **99 minutes**
  while the vast majority of TV shows have only **1 season** (1,608 of 2,410 shows) — a "long tail" of renewed
  shows is rare.
- Rows went from 7,787 to **7,777** after dropping the 10 rows with missing `date_added`.


## ***4. Data Vizualization, Storytelling & Experimenting with charts : Understand the relationships between variables***

#### Chart - 1

In [ ]:
# Chart - 1 visualization code
type_counts = df['type'].value_counts()

plt.figure(figsize=(6, 6))
plt.pie(type_counts, labels=type_counts.index, autopct='%1.1f%%',
        colors=[NETFLIX_RED, NETFLIX_BLACK], explode=(0.03, 0.03), startangle=90)
plt.title('Share of Content: Movies vs TV Shows')
plt.show()
print(type_counts)


##### 1. Why did you pick the specific chart?

A pie chart is the clearest way to show the overall two-category split (Movie vs. TV Show) as a share of the whole catalog.

##### 2. What is/are the insight(s) found from the chart?

Movies make up **69.1%** (5,377 titles) of the catalog vs. **30.9%** (2,410 titles) for TV Shows — Netflix is still primarily a movie platform in absolute terms, even though (as Chart 2 shows) the trend is shifting.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** Knowing the current mix helps set a baseline for tracking catalog strategy over time and helps content teams understand which format currently dominates recommendations.

#### Chart - 2

In [ ]:
# Chart - 2 visualization code
trend = df.groupby(['year_added', 'type']).size().unstack(fill_value=0)

trend.plot(kind='line', marker='o', color=[NETFLIX_RED, NETFLIX_BLACK], figsize=(10, 5))
plt.title('Titles Added to Netflix Per Year, by Type')
plt.ylabel('Number of Titles Added')
plt.xlabel('Year Added')
plt.show()

tv_share = (trend['TV Show'] / (trend['Movie'] + trend['TV Show'])).round(3)
print("TV Show share of additions by year:\n", tv_share)


##### 1. Why did you pick the specific chart?

A line chart is best for showing a trend over a continuous time axis (year), and using two lines lets us directly compare the growth rates of Movies vs. TV Shows.

##### 2. What is/are the insight(s) found from the chart?

TV Show's share of yearly additions rose steadily: **25.5% in 2018 → 30.5% in 2019 → 34.7% in 2020**, while movie additions grew more slowly and dipped in absolute count in 2020. This directly confirms Flixable's 2018 finding and shows the trend continued afterward (2021 is a partial year in this dataset, so its low counts are a cutoff artifact, not a decline).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This confirms Netflix has been increasingly prioritizing TV shows — useful for content planning teams to keep pace with this shift, and a caution for movie-focused production partners about a shrinking relative share of catalog additions.

#### Chart - 3

In [ ]:
# Chart - 3 visualization code
country_series = df['country'].str.split(',').explode().str.strip()
top_countries = country_series[~country_series.isin(['', 'Unknown'])].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_countries.values, y=top_countries.index, palette='Reds_r')
plt.title('Top 10 Countries Producing Netflix Content')
plt.xlabel('Number of Titles')
plt.show()
print(top_countries)


##### 1. Why did you pick the specific chart?

A horizontal bar chart works best for ranking a moderate number of categories (countries) by count, with country names easy to read on the y-axis.

##### 2. What is/are the insight(s) found from the chart?

The **United States (3,297 titles)** dominates by a wide margin, followed by **India (990)** and the **United Kingdom (723)** — together these three countries account for roughly two-thirds of all country-tagged content.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This highlights where Netflix's catalog is concentrated and where there may be room to diversify — e.g., investing further in high-population, currently under-represented markets could open new subscriber growth.

#### Chart - 4

In [ ]:
# Chart - 4 visualization code
genre_series = df['listed_in'].str.split(',').explode().str.strip()
top_genres = genre_series.value_counts().head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_genres.values, y=top_genres.index, palette='mako')
plt.title('Top 15 Genres on Netflix')
plt.xlabel('Number of Titles')
plt.show()
print(top_genres)


##### 1. Why did you pick the specific chart?

A horizontal bar chart clearly ranks genre popularity by title count across many genre tags.

##### 2. What is/are the insight(s) found from the chart?

**International Movies (2,437)**, **Dramas (2,106)**, and **Comedies (1,471)** are the three most common genre tags, and international content (Movies + TV Shows) appears very frequently overall — reflecting Netflix's global content strategy.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** Genre popularity guides which genres are safest to keep investing in, and also flags which niche genres (near the bottom of the top-15) might be worth expanding for differentiation.

#### Chart - 5

In [ ]:
# Chart - 5 visualization code
plt.figure(figsize=(10, 5))
order = df['rating'].value_counts().index
sns.countplot(data=df, y='rating', order=order, palette='rocket')
plt.title('Distribution of Content Ratings')
plt.xlabel('Number of Titles')
plt.show()
print(df['rating'].value_counts())


##### 1. Why did you pick the specific chart?

A count plot (bar chart of category counts) is the standard way to show the distribution of a categorical variable like content rating.

##### 2. What is/are the insight(s) found from the chart?

**TV-MA (2,863)** and **TV-14 (1,931)** are by far the most common ratings — together nearly 62% of the catalog is rated for mature/teen audiences, while family-friendly ratings (`G`, `TV-Y`, `TV-Y7`) make up a small fraction.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Mixed impact.** This is positive for Netflix's core adult/young-adult audience, but signals a potential gap in family-friendly content if Netflix wants to grow its family-plan subscriber base.

#### Chart - 6

In [ ]:
# Chart - 6 visualization code
movie_durations = df[df['type'] == 'Movie']['duration_int'].dropna()

plt.figure(figsize=(9, 5))
sns.histplot(movie_durations, bins=30, color=NETFLIX_RED, kde=True)
plt.title('Distribution of Movie Duration (minutes)')
plt.xlabel('Duration (minutes)')
plt.show()
print(movie_durations.describe())


##### 1. Why did you pick the specific chart?

A histogram (with a KDE overlay) is the right choice for showing the shape/spread of a continuous numeric variable such as duration.

##### 2. What is/are the insight(s) found from the chart?

Movie duration is roughly bell-shaped and centered around **~99 minutes** (mean 99.3, median 98), with the bulk of movies falling between 86 and 114 minutes (IQR) — very few movies run under 60 or over 180 minutes.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** Knowing the typical runtime helps set viewer expectations and can guide production/licensing decisions if Netflix wants more short-form or long-form content for specific audience segments.

#### Chart - 7

In [ ]:
# Chart - 7 visualization code
tv_seasons = df[df['type'] == 'TV Show']['duration_int'].dropna()

plt.figure(figsize=(9, 5))
sns.countplot(x=tv_seasons, color=NETFLIX_BLACK)
plt.title('Number of Seasons per TV Show')
plt.xlabel('Seasons')
plt.show()
print(tv_seasons.value_counts().sort_index())


##### 1. Why did you pick the specific chart?

A count plot suits a small-range discrete numeric variable like number of seasons, showing exactly how many shows fall into each season count.

##### 2. What is/are the insight(s) found from the chart?

The vast majority of TV shows (**1,608 out of 2,410, ~67%**) have only **1 season**, and the count drops off sharply after that — very few shows are renewed for 5+ seasons.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This suggests most TV shows are either mini-series/limited runs or new shows awaiting renewal decisions — useful context for retention teams evaluating whether under-performing single-season shows should be renewed or replaced.

#### Chart - 8

In [ ]:
# Chart - 8 visualization code
top_directors = df[df['director'] != 'Unknown']['director'].value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_directors.values, y=top_directors.index, palette='Reds_r')
plt.title('Top 10 Directors by Number of Titles on Netflix')
plt.xlabel('Number of Titles')
plt.show()
print(top_directors)


##### 1. Why did you pick the specific chart?

A horizontal bar chart ranks the (small) set of most-featured directors clearly by title count.

##### 2. What is/are the insight(s) found from the chart?

The directing duo **Raúl Campos & Jan Suter** top the list with 18 titles (mostly stand-up comedy specials), followed by **Marcus Raboy (16)** and **Jay Karas (14)** — both also comedy-special directors — showing Netflix's heavy investment in stand-up comedy content.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This identifies Netflix's most productive recurring content partners, useful for negotiating future deals or understanding which directors are strongly associated with a successful content category (stand-up comedy).

#### Chart - 9

In [ ]:
# Chart - 9 visualization code
cast_series = df[df['cast'] != 'Unknown']['cast'].str.split(',').explode().str.strip()
top_cast = cast_series.value_counts().head(10)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_cast.values, y=top_cast.index, palette='mako')
plt.title('Top 10 Most Frequently Credited Cast Members')
plt.xlabel('Number of Titles')
plt.show()
print(top_cast)


##### 1. Why did you pick the specific chart?

A horizontal bar chart is again well-suited to ranking a set of individual cast members by appearance count after exploding the comma-separated `cast` column.

##### 2. What is/are the insight(s) found from the chart?

**Anupam Kher (42 titles)**, **Shah Rukh Khan (35)**, and several other Bollywood actors dominate the top of the list — reflecting how large a share of Netflix's catalog is sourced from the Indian film industry.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This surfaces Netflix's most bankable recurring talent by region, useful for regional content marketing (e.g., promoting an actor's back-catalog when a new title featuring them is added).

#### Chart - 10

In [ ]:
# Chart - 10 visualization code
plt.figure(figsize=(9, 5))
sns.histplot(df['release_year'], bins=30, color=NETFLIX_RED)
plt.title('Distribution of Release Year')
plt.xlabel('Release Year')
plt.show()
print(df['release_year'].describe())


##### 1. Why did you pick the specific chart?

A histogram is the standard choice for visualizing the distribution/skew of a numeric year variable across the whole catalog.

##### 2. What is/are the insight(s) found from the chart?

The distribution is heavily skewed toward recent years — the median release year is **2017** and 75% of titles were released in **2013 or later** — with a long thin tail of older classics stretching back to 1925.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This confirms Netflix's catalog is weighted toward recent content, which is useful context when interpreting other trends (e.g., genre popularity is partly driven by what's been made recently, not just licensed classics).

#### Chart - 11

In [ ]:
# Chart - 11 visualization code
month_counts = df['month_added'].value_counts().sort_index()
month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']

plt.figure(figsize=(10, 5))
sns.barplot(x=[month_names[int(m)-1] for m in month_counts.index], y=month_counts.values, color=NETFLIX_RED)
plt.title('Titles Added to Netflix by Month (All Years Combined)')
plt.ylabel('Number of Titles Added')
plt.show()
print(month_counts)


##### 1. Why did you pick the specific chart?

A bar chart across the 12 calendar months is the clearest way to check for seasonality in when Netflix adds new content.

##### 2. What is/are the insight(s) found from the chart?

**October, November, and December** see the highest volume of additions (785, 738, 833 respectively), suggesting Netflix loads up its catalog ahead of the holiday season, while **February** sees the fewest additions (472).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** Marketing and subscription-retention teams can use this seasonality to plan promotional campaigns and manage subscriber churn around the months with the biggest content drops.

#### Chart - 12

In [ ]:
# Chart - 12 visualization code
top5_countries = df['primary_country'].value_counts().head(5).index
subset = df[df['primary_country'].isin(top5_countries)]
ct = pd.crosstab(subset['primary_country'], subset['type'], normalize='index')

ct.plot(kind='bar', stacked=True, color=[NETFLIX_RED, NETFLIX_BLACK], figsize=(9, 5))
plt.title('Movie vs TV Show Mix in Top 5 Countries')
plt.ylabel('Proportion of Titles')
plt.xticks(rotation=0)
plt.show()
print(ct.round(3))


##### 1. Why did you pick the specific chart?

A stacked, normalized bar chart is ideal for comparing the internal Movie/TV Show composition of several countries side by side.

##### 2. What is/are the insight(s) found from the chart?

**India is 92.4% Movies** — the most movie-heavy of the top countries — while the **United Kingdom is the most TV-Show-heavy (40.8%)** among named countries. The United States sits roughly in between (73.0% Movies).

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** Country-specific content-type ratios can inform regional acquisition strategy — e.g. Netflix could look to grow its TV-show slate in India specifically, where movies overwhelmingly dominate.

#### Chart - 13

In [ ]:
# Chart - 13 visualization code
rating_type = pd.crosstab(df['rating'], df['type'])

rating_type.plot(kind='bar', color=[NETFLIX_RED, NETFLIX_BLACK], figsize=(11, 5))
plt.title('Content Rating by Type (Movie vs TV Show)')
plt.ylabel('Number of Titles')
plt.xticks(rotation=45)
plt.show()
print(rating_type)


##### 1. Why did you pick the specific chart?

A grouped bar chart lets us compare, rating by rating, how many Movies vs. TV Shows fall into each rating category.

##### 2. What is/are the insight(s) found from the chart?

Ratings like **PG, PG-13, R, G, and NC-17 are used almost exclusively for Movies** (TV Shows use `TV-`-prefixed ratings instead), and within the TV-rating system, `TV-MA` and `TV-14` are common for both types, while `TV-Y`/`TV-Y7` (children's ratings) skew more toward TV Shows — consistent with kids' programming being predominantly episodic.

##### 3. Will the gained insights help creating a positive business impact?
Are there any insights that lead to negative growth? Justify with specific reason.

**Yes, positive impact.** This confirms the two content types use largely separate rating systems, a detail worth accounting for in any future modeling that treats `rating` as a single unified feature.

#### Chart - 14 - Correlation Heatmap

In [ ]:
# Correlation Heatmap visualization code
numeric_cols = ['release_year', 'year_added', 'month_added', 'duration_int', 'content_age_at_add']
corr = df[numeric_cols].corr()

plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='RdBu_r', center=0, vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Numeric Features')
plt.show()


##### 1. Why did you pick the specific chart?

A heatmap is the standard way to visualize pairwise correlation strength across several numeric features at once, using color to make patterns easy to spot.

##### 2. What is/are the insight(s) found from the chart?

As expected, `release_year` and `content_age_at_add` are almost perfectly negatively correlated (**-0.99**, since content age is derived from release year), and `duration_int` has a mild negative correlation with `release_year` (**-0.24**), hinting that older movies in the catalog tend to run slightly longer. `month_added` shows negligible correlation with everything else, confirming there's no seasonal bias in duration or release year.

#### Chart - 15 - Pair Plot

In [ ]:
# Pair Plot visualization code
sample_df = df[numeric_cols + ['type']].dropna().sample(n=min(1000, len(df)), random_state=42)

sns.pairplot(sample_df, hue='type', palette=[NETFLIX_RED, NETFLIX_BLACK], diag_kind='hist', plot_kws={'alpha': 0.5, 's': 15})
plt.suptitle('Pair Plot of Numeric Features by Content Type', y=1.02)
plt.show()


##### 1. Why did you pick the specific chart?

A pair plot (scatterplot matrix) shows every pairwise relationship between numeric features at once, and coloring by `type` reveals whether Movies and TV Shows occupy different regions of that feature space.

##### 2. What is/are the insight(s) found from the chart?

TV Shows and Movies separate cleanly on the `duration_int` axis (seasons vs. minutes are different unit scales, as expected), and TV Shows cluster in a narrower `release_year` range (mostly post-2015) compared to Movies, which have a longer tail of older releases — reinforcing that Netflix's TV catalog is a comparatively newer addition to the platform.

## **5. Solution to Business Objective**

#### What do you suggest the client to achieve Business Objective ?
Explain Briefly.

Based on the EDA above, we recommend Netflix:

1. **Keep growing the TV Show slate**, especially in markets like the UK where TV already outpaces movies, since the
   overall trend (Chart 2) shows TV's share of yearly additions climbing from ~25% (2018) to ~35% (2020).
2. **Diversify beyond the top 3 countries** (US, India, UK ≈ two-thirds of the catalog) by expanding local content
   in large, currently under-represented markets to support international subscriber growth.
3. **Add more family-friendly (G/PG/TV-Y/TV-Y7) content**, since mature ratings (TV-MA, TV-14) dominate the catalog
   at nearly 62% — a gap for households wanting more kid-safe options.
4. **Grow India's TV Show output specifically** — India is 92.4% movies, the most movie-skewed of the top
   countries, suggesting room to build out episodic content for that market.
5. **Plan major content drops around Oct-Dec**, since that's already the platform's heaviest addition season —
   aligning marketing campaigns with this pattern can maximize engagement.


# **Conclusion**

This EDA of 7,787 Netflix titles confirmed the trend first reported by Flixable in 2018: Netflix has been
steadily increasing its TV Show additions relative to Movies, with TV's share of yearly additions rising from about
25% in 2018 to nearly 35% in 2020. The catalog remains heavily concentrated in the United States, India, and the
United Kingdom, skews toward mature content ratings, and sees its heaviest content drops in the last quarter of the
year. Movie runtimes cluster tightly around 99 minutes, while the vast majority of TV shows (67%) have only ever
had a single season. These insights give Netflix's content, regional licensing, and marketing teams concrete,
data-backed levers — from where to expand production, to which content gaps to close, to when to schedule major
catalog updates.


### ***Hurrah! You have successfully completed your EDA Capstone Project !!!***